# SpectraRestore — Google Colab

**KLA PS01 · SEMICON India Hackathon 2026**  
**Team:** ChipSync · SRM Institute of Science And Technology  
Joint Blind Denoising + 2× Super-Resolution for SEM Inspection Images (`NAFNet-SR2×`)

### Workflow Overview
1. **Runtime:** Set Hardware Accelerator to **GPU** (T4 / A100 / L4) under `Runtime → Change runtime type`.
2. **Run All:** Directly clones the public repository from GitHub, installs requirements, executes smoke tests, trains the model, performs evaluation, and generates validation benchmark tables and visual error heatmaps for Slide 6.

## 0 · Check GPU Acceleration

In [ ]:
!nvidia-smi
import torch
print('PyTorch version:', torch.__version__, '| CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))
    print('VRAM:', f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')

## 1 · Clone Repository from GitHub

Clones the public repository directly into the Colab environment.

In [ ]:
import os
from pathlib import Path

REPO_URL = 'https://github.com/Charan-suresh/SpectraRestore.git'
PROJECT = Path('/content/SpectraRestore')

if (PROJECT / 'src' / 'model.py').is_file():
    print(f'Project already present at {PROJECT} — pulling latest updates...')
    %cd {PROJECT}
    !git pull origin main
else:
    print(f'Cloning repository from {REPO_URL}...')
    !git clone {REPO_URL} {PROJECT}
    %cd {PROJECT}

assert (PROJECT / 'src' / 'model.py').is_file(), 'Setup failed — repository cloning was unsuccessful.'
print(f'\nCurrent Working Directory: {Path.cwd()}')

## 2 · Install Dependencies & Verify Architecture

In [ ]:
%pip install -q -r requirements.txt

import sys
from pathlib import Path
PROJECT = Path.cwd()
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from src.model import build_model
m_default = build_model('default')
m_fast = build_model('fast')
print(f'NAFNet-SR2x Default Model: {m_default.num_params()/1e6:.2f}M parameters')
print(f'NAFNet-SR2x Fast Model:    {m_fast.num_params()/1e6:.2f}M parameters')

## 3 · Dataset Setup & Verification

Upload or unpack your dataset into `data/` with the following structure:
```
data/
  train/
    degraded/   (128x128 / 256x256 images: png, tif, npy)
    gt/         (256x256 / 512x512 clean target images)
  val/
    degraded/
    gt/
```

In [ ]:
from pathlib import Path

data_dir = Path('data')
data_dir.mkdir(exist_ok=True)

train_deg = data_dir / 'train' / 'degraded'
val_deg = data_dir / 'val' / 'degraded'

if train_deg.is_dir() or val_deg.is_dir():
    print('Dataset detected in data/:')
    !find data -type f | sed 's|/[^/]*$||' | sort | uniq -c | sort -rn | head -10
else:
    print('Ready for dataset. Upload or unpack your images into data/train/ and data/val/')

## 4 · Quick System & Pipeline Smoke Test

Verifies model initialization, loss functions, synthetic tensor forward/backward, and inference round-trip.

In [ ]:
!python scripts/smoke_test.py

## 5 · Train SpectraRestore (NAFNet-SR2×)

- **Preset:** `default` (~29M params) or `fast` (~15M params)
- **Loss:** Charbonnier (1.00) + SSIM (0.20) + FFT-L1 (0.05) + LPIPS (0.10 after warmup)
- Checkpoints are saved locally to `weights/` (`best.pt`, `last_ema.pt`, `ckpt_*.pt`).

In [ ]:
from pathlib import Path

PRESET = 'default'       # default (~29M) | fast (~15M) | large (~65M)
BATCH = 4                # 4 for T4 GPU, 8 for A100/L4
ITERS = 50000            # Initial pass (or 200000 for full convergence)
GT_CROP = 256

weights_dir = Path('weights')
weights_dir.mkdir(exist_ok=True)

# Resume from local checkpoint if available
resume = ''
ckpts = sorted(weights_dir.glob('ckpt_*.pt'))
if ckpts:
    resume = f' --resume {ckpts[-1]}'
    print(f'Resuming from latest checkpoint: {ckpts[-1]}')
elif (weights_dir / 'best.pt').is_file():
    resume = ' --resume weights/best.pt'
    print('Resuming from weights/best.pt')

cmd = f'''python -m src.train \\
  --data_root data \\
  --preset {PRESET} \\
  --batch_size {BATCH} \\
  --iters {ITERS} \\
  --gt_crop {GT_CROP} \\
  --num_workers 2 \\
  --val_every 1000 \\
  --save_every 2000 \\
  --log_every 50 \\
  --out_dir weights{resume}
'''
print('Executing training command:\n', cmd)
!{cmd}

## 6 · Run Standalone Inference (KLA evaluate.py)

Restores any directory of degraded SEM images and preserves native filenames for evaluation matching.

In [ ]:
from pathlib import Path

INPUT = 'data/val/degraded'          # Or KLA released test directory
OUTPUT = 'outputs/val_restored'

!python evaluate.py --input_dir {INPUT} --output_dir {OUTPUT} --weights weights/best.pt

## 7 · Validation Benchmark & Slide 6 Results Table

Computes exact SSIM, pSNR, LPIPS, and inference latency for both the **Degraded Input Baseline** and **SpectraRestore Output** on the held-out validation split.

In [ ]:
!python scripts/benchmark_val.py --data_root data --weights weights/best.pt

## 8 · Visual Evidence & Error Heatmaps (Slide 6 Matching)

Displays 4-panel visual comparison: `[Degraded Input] → [Our Restoration] → [Ground Truth] → [|Error| Heatmap]`.

In [ ]:
import random
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from evaluate import load_gray

out_dir = Path('outputs/val_restored')
deg_dir = Path('data/val/degraded')
gt_dir  = Path('data/val/gt')

restored_files = sorted([p for p in out_dir.rglob('*') if p.is_file()])
if restored_files:
    sample = random.choice(restored_files)
    stem = sample.stem
    
    deg_path = next(deg_dir.glob(stem + '.*'), None)
    gt_path = next(gt_dir.glob(stem + '.*'), None) if gt_dir.exists() else None
    
    deg_img = load_gray(deg_path) if deg_path else None
    rest_img = load_gray(sample)
    gt_img = load_gray(gt_path) if gt_path else None
    
    num_cols = 4 if gt_img is not None else 2
    fig, axs = plt.subplots(1, num_cols, figsize=(4 * num_cols, 4), dpi=150)
    
    axs[0].imshow(np.clip(deg_img, 0, 1), cmap='gray')
    axs[0].set_title('Degraded Input', fontsize=12, fontweight='bold')
    axs[0].axis('off')
    
    axs[1].imshow(np.clip(rest_img, 0, 1), cmap='gray')
    axs[1].set_title('Our Restoration', fontsize=12, fontweight='bold')
    axs[1].axis('off')
    
    if gt_img is not None:
        axs[2].imshow(np.clip(gt_img, 0, 1), cmap='gray')
        axs[2].set_title('Ground Truth', fontsize=12, fontweight='bold')
        axs[2].axis('off')
        
        # Absolute error heatmap
        err = np.abs(np.clip(rest_img, 0, 1) - np.clip(gt_img, 0, 1))
        im_err = axs[3].imshow(err, cmap='turbo', vmin=0.0, vmax=1.0)
        axs[3].set_title('|Error| Heatmap', fontsize=12, fontweight='bold')
        axs[3].axis('off')
        cbar = plt.colorbar(im_err, ax=axs[3], fraction=0.046, pad=0.04)
        cbar.ax.tick_params(labelsize=9)
    
    plt.suptitle(f'Restoration Evidence: {sample.name}', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print('No restored outputs found. Run evaluation (Cell 6) first.')